# parameter-subclass-of-tensor — ex3: freeze(module, prefix): flip requires_grad=False on every Parameter whose name starts with prefix

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parameter-subclass-of-tensor`. Running the final beacon cell reports progress against the `Backprop: Parameter subclasses Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter subclasses Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-subclass-of-tensor`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-subclass-of-tensor"
DD_SUBTOPIC = "Backprop: Parameter subclasses Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `freeze(module, prefix)` — mutate `requires_grad` on selected Parameters

Ex1 defined `Parameter` as a typed subclass. Ex2 filtered for trainable params via `isinstance(_, Parameter)`. The third facet is the operational use of the type tag: walk parameters() and FLIP `requires_grad` based on a name prefix — the standard frozen-backbone pattern for fine-tuning.

```python
def freeze(module, prefix):
    """Set requires_grad=False on every Parameter whose dotted name starts with prefix."""
    for name, p in module.parameters():
        if name.startswith(prefix):
            p.requires_grad = False
```

**Why the type tag is what matters, not requires_grad.** A Parameter with `requires_grad=False` is STILL a Parameter — `trainable_params` still yields it (ex2's load-bearing invariant). What changes is the gradient-flow gate: the wrapper's any-rg-input check excludes rg=False parents, so no gradient ever reaches the frozen param.

**Why mutate in place, not return a new module.** Modules carry state — buffers, configs, registered hooks. Cloning the whole module to flip one flag would be wasteful and break any external reference to the module. PyTorch's standard idiom is exactly the same: `for p in model.encoder.parameters(): p.requires_grad = False`.

**The dotted prefix matches naming from ex2.** `freeze(model, 'fc1.')` freezes everything under `fc1` — `fc1.weight`, `fc1.bias`. `freeze(model, 'fc1.weight')` freezes ONLY the weight. The prefix is a string match, not a glob, so the trailing `.` matters.

### Exercise 3 — freeze(module, prefix): flip requires_grad=False on every Parameter whose name starts with prefix

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the freeze-by-prefix pattern: walk a module's parameters() and set requires_grad=False on every Parameter whose dotted name starts with a given prefix, leaving the Parameter type tag (and thus trainable_params filtering) intact.
> Keywords: freeze, fine-tune, prefix, requires-grad, parameter
> ```

**KCs targeted:** `freeze-by-name-prefix`, `type-tag-stable-under-rg-flip`

We've given you `Parameter` (subclass of MiniTensor with `requires_grad=True` default), a `Module` base class, and a `parameters(self)` generator that yields `(dotted_name, Parameter)` for every Parameter on the module (recursively).

Implement `ex3_freeze(module, prefix)`. Walk `module.parameters()` and set `p.requires_grad = False` on every Parameter whose yielded `dotted_name` starts with `prefix` (use `str.startswith`).

Behavior:

- Mutate Parameters in place (no return value needed; return `None`).
- `prefix` is a literal string match via `startswith`, NOT a regex or glob.
- Parameters whose name does NOT start with `prefix` are untouched.
- After the call, the frozen Parameters are STILL Parameters (type unchanged) — they just have `requires_grad=False`.

Examples:

```
model = MLPWithEncoder()  # has 'encoder.fc1.weight', 'encoder.fc1.bias',
                          #      'head.weight', 'head.bias'
freeze(model, 'encoder.')
  → 'encoder.fc1.weight' and 'encoder.fc1.bias' now rg=False
  → 'head.weight' and 'head.bias' still rg=True

freeze(model, 'encoder.fc1.weight')
  → only that specific weight is frozen; bias stays trainable
```

Constraints:
- Use `for name, p in module.parameters(): ...` — do NOT re-implement the walk.
- Do NOT change `p.array` or `p.recipe` — only flip `requires_grad`.
- `parameters()` STILL yields the frozen params after freezing (they're still Parameters; only the flag changed).

In [ ]:
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)


class Module:
    def parameters(self):
        """DFS yield of (dotted_name, Parameter) leaves."""
        for name, val in self.__dict__.items():
            if isinstance(val, Parameter):
                yield name, val
            elif isinstance(val, Module):
                for sub_name, sub_val in val.parameters():
                    yield f'{name}.{sub_name}', sub_val


def ex3_freeze(module, prefix: str) -> None:
    """Set requires_grad=False on every Parameter whose dotted name starts with prefix."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # === Single-level model: freeze one specific param ===
        class Linear(Module):
            def __init__(self):
                self.weight = Parameter(t.randn(4, 3))
                self.bias = Parameter(t.zeros(4))

        lin = Linear()
        assert lin.weight.requires_grad is True
        assert lin.bias.requires_grad is True

        ex3_freeze(lin, 'weight')
        assert lin.weight.requires_grad is False, 'weight must be frozen'
        assert lin.bias.requires_grad is True, (
            f'bias must stay trainable; rg={lin.bias.requires_grad}'
        )

        # === Two-level model: freeze a subtree by prefix ===
        class MLPWithEncoder(Module):
            def __init__(self):
                self.encoder = Linear()
                self.head = Linear()

        model = MLPWithEncoder()
        # All four params should start trainable.
        for name, p in model.parameters():
            assert p.requires_grad is True, f'{name} should start trainable'

        ex3_freeze(model, 'encoder.')
        flags = {name: p.requires_grad for name, p in model.parameters()}
        assert flags['encoder.weight'] is False
        assert flags['encoder.bias'] is False
        assert flags['head.weight'] is True
        assert flags['head.bias'] is True

        # === Frozen Parameters are STILL Parameters (type unchanged) ===
        assert isinstance(model.encoder.weight, Parameter), (
            'freeze must not change the type — only the flag'
        )
        assert isinstance(model.encoder.bias, Parameter)

        # === Frozen Parameters are still yielded by parameters() ===
        names_after = [n for n, _ in model.parameters()]
        assert names_after == ['encoder.weight', 'encoder.bias',
                               'head.weight', 'head.bias'], (
            f'parameters() must still yield all four; got {names_after}'
        )

        # === Specific-param freeze ===
        model2 = MLPWithEncoder()
        ex3_freeze(model2, 'encoder.weight')
        # Only encoder.weight should be frozen.
        flags = {name: p.requires_grad for name, p in model2.parameters()}
        assert flags['encoder.weight'] is False
        assert flags['encoder.bias'] is True, (
            'specific-param freeze must not affect siblings'
        )
        assert flags['head.weight'] is True
        assert flags['head.bias'] is True

        # === Empty prefix freezes EVERYTHING (startswith('') is always True) ===
        model3 = MLPWithEncoder()
        ex3_freeze(model3, '')
        for name, p in model3.parameters():
            assert p.requires_grad is False, f'{name} should be frozen'

        # === Non-matching prefix is a no-op ===
        model4 = MLPWithEncoder()
        ex3_freeze(model4, 'no_such_prefix.')
        for name, p in model4.parameters():
            assert p.requires_grad is True, f'{name} should still be trainable'

        # === Three-level nesting works ===
        class Block(Module):
            def __init__(self):
                self.inner = Linear()

        class Net(Module):
            def __init__(self):
                self.block = Block()
                self.classifier = Linear()

        net = Net()
        ex3_freeze(net, 'block.')
        flags = {n: p.requires_grad for n, p in net.parameters()}
        assert flags['block.inner.weight'] is False
        assert flags['block.inner.bias'] is False
        assert flags['classifier.weight'] is True
        assert flags['classifier.bias'] is True

        # === Return value is None ===
        model5 = MLPWithEncoder()
        ret = ex3_freeze(model5, 'head.')
        assert ret is None, f'freeze should return None, got {ret!r}'

        # === .array is untouched ===
        model6 = MLPWithEncoder()
        arr_before = model6.encoder.weight.array.clone()
        ex3_freeze(model6, 'encoder.')
        assert t.allclose(model6.encoder.weight.array, arr_before), (
            '.array must be untouched — only requires_grad flips'
        )
        print('ex3 ✓')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_freeze(module, prefix):
    for name, p in module.parameters():
        if name.startswith(prefix):
            p.requires_grad = False
```

**`startswith` is the canonical prefix test.** It returns True for an empty prefix (a useful 'freeze all' shortcut) and False for non-prefixes. No glob/regex needed — we WANT the strict literal match, since prefixes ARE the natural hierarchy encoded by dotted names.

**Why mutate, not return a new module.** Modules are stateful — they carry buffers, configs, registered hooks. Cloning to flip one flag wastes memory and breaks any external reference. PyTorch's idiom is exactly this in-place mutation: `for p in model.encoder.parameters(): p.requires_grad = False`.

**Why the type-tag persists.** A frozen Parameter is STILL a Parameter. The type identifies 'this is trainable state, possibly frozen', distinct from 'this is a buffer (running mean)' or 'this is an incidental tensor'. `trainable_params` from ex2 filters by TYPE, not by `requires_grad` — frozen Parameters still show up in state_dict / checkpointing, just don't receive gradient updates.

**Why this is the third facet.** Ex1 defined the type. Ex2 used the type as a filter. Ex3 toggles the orthogonal `requires_grad` flag on instances found via that filter — showing the type-tag and the gradient-flag are independent axes. Same Parameter, different temporary state.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()